# 03 · Shared comparison: City A vs City B, models, business answer

**PA04 · Textile storage facility (humid vs. dry regimes) · Guimarães (City A) vs Viana do Castelo (City B) · T4 Humid vs Dry Regime**

* **Business context:** textile storage facility (humid vs. dry regimes)
* **Locations:** Guimarães vs Viana do Castelo
* **Question:** How do humid and dry regimes differ between candidate warehouse locations?

| | Student | Number | Primary city |
|---|---|---|---|
| Student A | Khadija Belhamra | 20230007 | City A: Guimarães |
| Student B | Victoriya Mokhovikova | 20230796 | City B: Viana do Castelo |

The shared notebook that comes **last**: it pairs the work of both students. It needs the shared protocol (`prepare_frames`), the profile of both cities (`profiling.profile_city`, the same functions as notebooks 01 / 02) and the official results each city notebook saved (`results/metrics_<city>.csv`, `results/history_<city>.csv`).

| Step | Content |
|---|---|
| 1 | Load both cities and both students' results |
| 2 | Key numbers side by side |
| 3 | Humid regime: frequency, persistence, timing |
| 4 | Model comparison (both representations, both cities, persistence reference) |
| 5 | Performance over time |
| 6 | Cost: prediction and update time |
| 7 | Business answer and technical recommendation |

## 1. Load both cities and both students' results

In [ ]:
import sys
from pathlib import Path

# Make src/ importable whatever folder Jupyter was started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import plots
import profiling
from data import prepare_frames
from evaluate import load_results
from utils import load_config

plots.set_style()
cfg = load_config()
prepared = prepare_frames(cfg)
NAMES = {c["key"]: c["name"] for c in cfg["cities"]}
profiles = {k: profiling.profile_city(f, prepared["tau"], cfg) for k, f in prepared["frames"].items()}

metrics, history = load_results(cfg)        # saved by 01_cityA_guimaraes and 02_cityB_viana_do_castelo
print(f"tau_H = {prepared['tau']:.2f} %  |  {len(metrics)} runs  |  "
      f"{metrics['n_eval'].iloc[0]} evaluation events per stream")

## 2. Key numbers side by side

In [ ]:
key = pd.DataFrame({NAMES[k]: p["key_numbers"] for k, p in profiles.items()})
key["B - A"] = key.iloc[:, 1] - key.iloc[:, 0]
key.round(3)

## 3. Humid regime: frequency, persistence, timing

Same τ_H for both cities, so the shares and episode lengths are directly comparable.

In [ ]:
fig = plots.monthly_humid_share(prepared, cfg)
plots.save(fig, cfg, "comparison_01_monthly_humid_share")
plt.show()

In [ ]:
pd.concat({NAMES[k]: p["persistence"] for k, p in profiles.items()}, axis=1).round(2)

In [ ]:
fig = plots.compare_episode_durations(profiles, cfg)
plots.save(fig, cfg, "comparison_02_humid_episode_durations")
plt.show()

In [ ]:
fig = plots.compare_hour_month(profiles, cfg)
plots.save(fig, cfg, "comparison_03_hour_month")
plt.show()

> 📝 **For the report:** where is the humid regime more frequent, more persistent, and at which times? Is the difference seasonal, daily, or both?

## 4. Model comparison

Same chronological history, same features and same models for both cities (Ch. 3.1). The dashed lines are the persistence rule ŷ_t = y_{t−1} of each city (`profiling.persistence_reference`, evaluation stream only).

In [ ]:
table = metrics.pivot_table(index="model", columns=["city", "representation"], values="macro_f1", sort=False)
table.columns = [f"{NAMES[c]} · {r}" for c, r in table.columns]
table.round(4)

In [ ]:
references = {k: p["persistence_reference"] for k, p in profiles.items()}
fig = plots.compare_models(metrics, references, cfg, "macro_f1")
plots.save(fig, cfg, "comparison_04_macro_f1")
plt.show()

In [ ]:
# Does the context of Representation B help? (B - A, per city and model)
gain = metrics.pivot_table(index="model", columns=["city", "representation"], values="macro_f1", sort=False)
pd.DataFrame({NAMES[c]: gain[(c, "B")] - gain[(c, "A")] for c in NAMES}).round(4)

In [ ]:
# Precision / Recall for the humid class (the event that matters for textile storage)
metrics.pivot_table(index="model", columns=["city", "representation"], values=["precision", "recall"],
                    sort=False).round(3)

## 5. Performance over time

Sliding-window Macro-F1 over the last `protocol.evaluation_window` events. Drops show periods where the models adapt to a changed regime.

In [ ]:
for rep in ("A", "B"):
    fig = plots.recent_performance(history, cfg, representation=rep)
    plots.save(fig, cfg, f"comparison_05_recent_macro_f1_{rep}")
    plt.show()

## 6. Cost: prediction and update time per event

In [ ]:
metrics.pivot_table(index="model", columns=["city", "representation"],
                    values=["pred_ms_per_event", "update_ms_per_event"], sort=False).round(3)

## 7. Business answer and technical recommendation

> 📝 **For the report: joint section.** Write this together (AI policy, Ch. 16).
> * **Business answer:** how do humid and dry regimes differ between Guimarães and Viana do Castelo, and what does it mean for a textile storage facility (environmental control, ventilation timing, risk of long humid episodes)?
> * **Technical recommendation:** which model and representation would you deploy, and why (Macro-F1, Recall on humid, gain over persistence, stability over time, cost)?
> * **Limitations:** one grid cell per city, three years, outdoor humidity only, τ_H estimated from one year.